# Day 15 — Treatment Assignment Evidence

## Project

EconoCausal — Dynamic Pricing via Double Machine Learning

## Objective

Investigate whether the available AmExpert 2019 Marketing Campaign datasets and their documentation provide sufficient evidence to define treatment assignment and customer-level campaign exposure.

## Tasks

* Review available campaign and training variables.
* Examine the evidence needed to establish treatment assignment.
* Distinguish campaign membership, coupon redemption, and actual treatment exposure.
* Investigate whether the available data supports a credible comparison between treated and untreated customers.
* Document unresolved questions and limitations.

## Expected Outcome

A structured assessment of the available treatment-assignment evidence and the additional information required before estimating causal effects.


In [1]:
import pandas as pd
import numpy as np

# Load relevant datasets
train = pd.read_csv("../data/raw/train.csv")
campaigns = pd.read_csv("../data/raw/campaign_data.csv")
customers = pd.read_csv("../data/raw/customer_demographics.csv")

# Inspect dataset dimensions
print("===== DATASET SHAPES =====")
print("Training data:", train.shape)
print("Campaign data:", campaigns.shape)
print("Customer demographics:", customers.shape)

# Inspect training columns
print("\n===== TRAINING DATA COLUMNS =====")
print(train.columns.tolist())

# Inspect campaign columns
print("\n===== CAMPAIGN DATA COLUMNS =====")
print(campaigns.columns.tolist())

# Inspect customer columns
print("\n===== CUSTOMER DATA COLUMNS =====")
print(customers.columns.tolist())

# Inspect training data
print("\n===== TRAINING DATA SAMPLE =====")
display(train.head())

# Inspect campaign data
print("\n===== CAMPAIGN DATA SAMPLE =====")
display(campaigns.head())

===== DATASET SHAPES =====
Training data: (78369, 5)
Campaign data: (28, 4)
Customer demographics: (760, 7)

===== TRAINING DATA COLUMNS =====
['id', 'campaign_id', 'coupon_id', 'customer_id', 'redemption_status']

===== CAMPAIGN DATA COLUMNS =====
['campaign_id', 'campaign_type', 'start_date', 'end_date']

===== CUSTOMER DATA COLUMNS =====
['customer_id', 'age_range', 'marital_status', 'rented', 'family_size', 'no_of_children', 'income_bracket']

===== TRAINING DATA SAMPLE =====


,id,campaign_id,coupon_id,customer_id,redemption_status
0,1,13,27,1053,0
1,2,13,116,48,0
2,6,9,635,205,0
3,7,13,644,1050,0
4,9,8,1017,1489,0



===== CAMPAIGN DATA SAMPLE =====


,campaign_id,campaign_type,start_date,end_date
0,24,Y,21/10/13,20/12/13
1,25,Y,21/10/13,22/11/13
2,20,Y,07/09/13,16/11/13
3,23,Y,08/10/13,15/11/13
4,21,Y,16/09/13,18/10/13


In [2]:
# Day 15: Treatment Assignment Evidence Audit

print("===== TRAINING DATA: POSSIBLE TREATMENT COLUMNS =====")

treatment_keywords = [
    "treatment",
    "treated",
    "assignment",
    "assigned",
    "exposure",
    "exposed",
    "delivery",
    "delivered",
    "eligible",
    "eligibility",
    "offer",
    "discount"
]

for dataset_name, df in [
    ("Training", train),
    ("Campaigns", campaigns),
    ("Customers", customers)
]:
    print(f"\n===== {dataset_name.upper()} DATASET =====")

    matching_columns = [
        col for col in df.columns
        if any(keyword in col.lower() for keyword in treatment_keywords)
    ]

    print("Potentially relevant columns:", matching_columns)

    print("All columns:", df.columns.tolist())


print("\n===== CAMPAIGN TYPE DISTRIBUTION =====")

if "campaign_type" in campaigns.columns:
    display(campaigns["campaign_type"].value_counts(dropna=False))


print("\n===== REDEMPTION STATUS DISTRIBUTION =====")

if "redemption_status" in train.columns:
    display(
        train["redemption_status"]
        .value_counts(dropna=False)
        .rename_axis("redemption_status")
        .reset_index(name="record_count")
    )


print("\n===== TREATMENT-ASSIGNMENT AUDIT COMPLETE =====")

===== TRAINING DATA: POSSIBLE TREATMENT COLUMNS =====

===== TRAINING DATASET =====
Potentially relevant columns: []
All columns: ['id', 'campaign_id', 'coupon_id', 'customer_id', 'redemption_status']

===== CAMPAIGNS DATASET =====
Potentially relevant columns: []
All columns: ['campaign_id', 'campaign_type', 'start_date', 'end_date']

===== CUSTOMERS DATASET =====
Potentially relevant columns: []
All columns: ['customer_id', 'age_range', 'marital_status', 'rented', 'family_size', 'no_of_children', 'income_bracket']

===== CAMPAIGN TYPE DISTRIBUTION =====


campaign_type
Y    22
X     6
Name: count, dtype: int64


===== REDEMPTION STATUS DISTRIBUTION =====


,redemption_status,record_count
0,0,77640
1,1,729



===== TREATMENT-ASSIGNMENT AUDIT COMPLETE =====


In [3]:
# Day 15: Campaign Membership and Treatment Comparison

print("===== CAMPAIGN TYPE SUMMARY =====")

if "campaign_type" in campaigns.columns:
    campaign_summary = (
        campaigns.groupby("campaign_type", dropna=False)
        .agg(
            campaign_count=("campaign_id", "nunique"),
            first_campaign_id=("campaign_id", "min"),
            last_campaign_id=("campaign_id", "max")
        )
        .reset_index()
    )

    display(campaign_summary)


print("\n===== CAMPAIGN COUNTS IN TRAINING DATA =====")

train_campaign_summary = (
    train.groupby("campaign_id")
    .agg(
        record_count=("campaign_id", "size"),
        unique_customers=("customer_id", "nunique"),
        unique_coupons=("coupon_id", "nunique"),
        redemption_rate=("redemption_status", "mean")
    )
    .reset_index()
)

display(train_campaign_summary.head(10))


print("\n===== CAMPAIGN TYPE AND REDEMPTION =====")

if "campaign_type" in campaigns.columns:
    merged = train.merge(
        campaigns[["campaign_id", "campaign_type"]],
        on="campaign_id",
        how="left",
        validate="many_to_one"
    )

    type_summary = (
        merged.groupby("campaign_type", dropna=False)
        .agg(
            record_count=("redemption_status", "size"),
            unique_customers=("customer_id", "nunique"),
            redemption_rate=("redemption_status", "mean")
        )
        .reset_index()
    )

    display(type_summary)

    print(
        "\nInterpretation: These are descriptive comparisons. "
        "They do not establish treatment assignment or causal effects."
    )

===== CAMPAIGN TYPE SUMMARY =====


,campaign_type,campaign_count,first_campaign_id,last_campaign_id
0,X,6,8,30
1,Y,22,1,29



===== CAMPAIGN COUNTS IN TRAINING DATA =====


,campaign_id,record_count,unique_customers,unique_coupons,redemption_rate
0,1,143,13,11,0.006993
1,2,768,48,16,0.006510
2,3,408,12,34,0.004902
3,4,972,81,12,0.007202
4,5,1827,166,11,0.003831
5,6,65,65,1,0.015385
6,7,1584,198,8,0.002525
7,8,22555,1076,208,0.008823
8,9,3168,176,18,0.007260
9,10,1723,123,14,0.004063



===== CAMPAIGN TYPE AND REDEMPTION =====


,campaign_type,record_count,unique_customers,redemption_rate
0,X,57684,1379,0.011026
1,Y,20685,800,0.004496



Interpretation: These are descriptive comparisons. They do not establish treatment assignment or causal effects.


In [4]:
# Day 15: Treatment Assignment Evidence Assessment

assessment = pd.DataFrame([
    {
        "assessment_area": "Explicit treatment variable",
        "finding": "No verified treatment variable has been established by the current analysis",
        "status": "Unresolved"
    },
    {
        "assessment_area": "Customer eligibility",
        "finding": "Customer eligibility for specific campaigns has not been verified",
        "status": "Insufficient evidence"
    },
    {
        "assessment_area": "Campaign delivery or exposure",
        "finding": "Campaign membership alone does not prove actual customer exposure",
        "status": "Unverified"
    },
    {
        "assessment_area": "Campaign type",
        "finding": "Campaign type is a descriptive characteristic unless its intervention meaning is established",
        "status": "Requires validation"
    },
    {
        "assessment_area": "Redemption outcome",
        "finding": "redemption_status records observed redemption and is a candidate outcome",
        "status": "Candidate identified"
    },
    {
        "assessment_area": "Treated-versus-control comparison",
        "finding": "A defensible treatment contrast has not yet been established",
        "status": "Unresolved"
    },
    {
        "assessment_area": "Causal interpretation",
        "finding": "Differences in observed redemption rates do not independently establish causal effects",
        "status": "Not established"
    },
    {
        "assessment_area": "Double Machine Learning readiness",
        "finding": "Treatment assignment and causal identification require further investigation",
        "status": "Not ready"
    }
])

display(assessment)

# Save assessment
assessment.to_csv(
    "../reports/day15_treatment_assignment_evidence.csv",
    index=False
)

print("Day 15 assessment saved successfully.")

,assessment_area,finding,status
0,Explicit treatment variable,No verified treatment variable has been establ...,Unresolved
1,Customer eligibility,Customer eligibility for specific campaigns ha...,Insufficient evidence
2,Campaign delivery or exposure,Campaign membership alone does not prove actua...,Unverified
3,Campaign type,Campaign type is a descriptive characteristic ...,Requires validation
4,Redemption outcome,redemption_status records observed redemption ...,Candidate identified
5,Treated-versus-control comparison,A defensible treatment contrast has not yet be...,Unresolved
6,Causal interpretation,Differences in observed redemption rates do no...,Not established
7,Double Machine Learning readiness,Treatment assignment and causal identification...,Not ready


Day 15 assessment saved successfully.
